In [1]:
import argparse
import json
import logging
import os
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import torch
import wandb
import yaml
from monai.utils import set_determinism
from sklearn.preprocessing import StandardScaler
from torch.utils.tensorboard import SummaryWriter

from src.data.data_loader import get_dataloader
from src.model.mil import MILModel3D
from src.train.train_pirads import train_epoch, val_epoch
from src.utils import save_pirads_checkpoint, setup_logging
from monai.metrics import Cumulative, CumulativeAverage
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm

In [2]:
import yaml
from argparse import Namespace

# 1. Read the YAML file into a dictionary
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/config/config_pirads_test.yaml', 'r') as file:
    config_dict = yaml.safe_load(file)

# 2. Convert the dictionary directly to a Namespace
args = Namespace(**config_dict)

In [3]:
args.project_dir = '/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate'
args.run_name = 'pirads_inf'
args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args.batch_size = 4
args.workers = 0

args.logdir = os.path.join(args.project_dir, "logs", args.run_name)
os.makedirs(args.logdir, exist_ok=True)
args.logfile = os.path.join(args.logdir, f"{args.run_name}.log")
setup_logging(args.logfile)
if args.device == torch.device("cpu"):
    args.amp = False

In [4]:
if args.device == torch.device("cuda"):
    torch.backends.cudnn.benchmark = True

model = MILModel3D(num_classes=args.num_classes, mil_mode=args.mil_mode)
start_epoch = 0
best_acc = 0.0
checkpoint = torch.load(args.checkpoint, map_location="cpu")
model.load_state_dict(checkpoint["state_dict"])

if "epoch" in checkpoint:
    start_epoch = checkpoint["epoch"]
if "best_acc" in checkpoint:
    best_acc = checkpoint["best_acc"]
logging.info(
    "=> loaded checkpoint %s (epoch %d) (bestacc %f)",
    args.checkpoint,
    start_epoch,
    best_acc,
)
cache_dir_ = os.path.join(args.logdir, "cache")
model.to(args.device)
params = model.parameters()

scaler = StandardScaler()
with open(os.path.join(args.project_dir, "dataset", "PICAI_cspca_updated_with_psa_updated_vol.json")) as f:
    dataset_json = json.load(f)
train_clinical = [i["psa"] for i in dataset_json["train"]]
_ = scaler.fit_transform(train_clinical)
args.psa_mean = scaler.mean_.tolist()
args.psa_std = scaler.scale_.tolist()

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [5]:
args.num_seeds = 1
args.epochs = 0
seed = 56
set_determinism(seed=seed)
valid_loader = get_dataloader(args, split='test')

preds_cumulative = Cumulative()
targets_cumulative = Cumulative()

model.eval()
with torch.no_grad():
    for idx, batch_data in tqdm(enumerate(valid_loader)):
        data = batch_data["image"].as_subclass(torch.Tensor).to(args.device)
        target = batch_data["pirads"].as_subclass(torch.Tensor).to(args.device)
        target = target.long()

        with torch.amp.autocast(device_type=str(args.device), enabled=args.amp):
            logits = model(data)

        data = data.to("cpu")
        target = target.to("cpu")
        logits = logits.to("cpu")
        pred = torch.argmax(logits, dim=1)

        preds_cumulative.extend(pred.detach().cpu())
        targets_cumulative.extend(target.detach().cpu())


        del data, target, logits
        # torch.cuda.empty_cache()

    # Calculate QWK metric (Quadratic Weigted Kappa) https://en.wikipedia.org/wiki/Cohen%27s_kappa
    preds_cumulative = preds_cumulative.get_buffer().cpu().numpy()
    targets_cumulative = targets_cumulative.get_buffer().cpu().numpy()




if os.path.exists(cache_dir_):

    shutil.rmtree(cache_dir_)


0it [00:00, ?it/s]You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implici

In [7]:
np.save('pred.npy', preds_cumulative)
np.save('target.npy', targets_cumulative)

In [6]:
pred = preds_cumulative
targets = targets_cumulative

In [7]:
targets_clipped = np.clip(targets, a_min = 2, a_max = 5)
target = targets_clipped - 2


In [8]:
qwk = cohen_kappa_score(
    target.astype(np.float64), pred.astype(np.float64)
)
qwk

0.23748254253920043

In [9]:
pred_mod = np.clip(pred, a_min = 0, a_max = 2)
target_mod = np.clip(target, a_min = 0, a_max = 2)
qwk = cohen_kappa_score(
    target_mod.astype(np.float64), pred_mod.astype(np.float64)
)
qwk

0.2053401834331301

In [10]:

preds_for_class_0 = pred[target == 0]


distribution = np.bincount(preds_for_class_0, minlength=4)

print("Counts for predicted classes [0, 1, 2, 3]:")
print(distribution)



Counts for predicted classes [0, 1, 2, 3]:
[ 27  88 113  14]


In [11]:
preds_for_class_0 = pred[target == 1]


distribution = np.bincount(preds_for_class_0, minlength=4)

print("Counts for predicted classes [0, 1, 2, 3]:")
print(distribution)

Counts for predicted classes [0, 1, 2, 3]:
[ 6 19 25 12]


In [12]:
preds_for_class_0 = pred[target == 2]


distribution = np.bincount(preds_for_class_0, minlength=4)

print("Counts for predicted classes [0, 1, 2, 3]:")
print(distribution)

Counts for predicted classes [0, 1, 2, 3]:
[ 12  34 129  49]


In [13]:
preds_for_class_0 = pred[target == 3]


distribution = np.bincount(preds_for_class_0, minlength=4)

print("Counts for predicted classes [0, 1, 2, 3]:")
print(distribution)

Counts for predicted classes [0, 1, 2, 3]:
[  3  16  87 182]
